In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-81a577e8-cf7a-44d6-b0cf-96f4204f9c1e;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 407ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
orders = spark.read.csv("s3a://last-mile-optimization-trusted/dataset-orders/final_table_orders_numeric.csv/", header=True, inferSchema=True)
weather = spark.read.csv("s3a://last-mile-optimization-trusted/dataset-weather/", header=True, inferSchema=True)

26/04/19 03:17:26 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
# Cria um novo DataFrame com as combinações únicas de latitude e longitude
df_locais_unicos = weather.select("latitude", "longitude").distinct()

df_locais_unicos.show()

# Conta a quantidade de registros (linhas) nessa nova tabela
total_locais = df_locais_unicos.count()

+------------+------------+
|    latitude|   longitude|
+------------+------------+
|-21.77972221|-47.07972221|
|-23.04194444|-45.52027777|
|-23.09972221|-48.94555555|
|-20.57999999|-47.37999999|
|     -22.415|-46.80527777|
|-23.84499999|-46.14305555|
|-23.85138888|-48.16444444|
|-24.96305555|-48.41638888|
|-21.45777777|-51.55222222|
|-21.92722221|-50.49027777|
|-22.37083332|-48.55722222|
|-22.35805555|-49.02888888|
|     -20.165|-50.59499999|
|-23.52333332|-46.86916666|
|-23.98138888|-48.88527777|
|-22.37249999|-50.97416666|
|-22.70277777|-47.62305554|
|-21.13305554|-48.84027777|
|-20.55888888|-48.54472221|
|-21.85555555|-48.66666666|
+------------+------------+
only showing top 20 rows



In [5]:
# print(total_locais)

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import math

def haversine(lat1, lon1, lat2, lon2):
    # Raio da Terra em km
    R = 6371.0
    
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    
    a = math.sin(dphi / 2)**2 + \
        math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Registrar como UDF
haversine_udf = F.udf(haversine, DoubleType())

In [7]:
# 1. Obter coordenadas únicas de Clientes e Vendedores para otimizar o processamento
customer_locs = orders.select("customer_geolocation_lat", "customer_geolocation_lng").distinct()
seller_locs = orders.select("seller_geolocation_lat", "seller_geolocation_lng").distinct()

# 2. Função para encontrar o vizinho mais próximo no DF de Weather
def find_nearest_weather(df_points, lat_col, lng_col, prefix):
    # Faz o cross join com a tabela de locais únicos do clima
    # Cuidado: Use apenas se df_locais_unicos for pequeno. 
    # Caso contrário, considere usar bibliotecas de indexação espacial (como Bhuvan ou GeoSpark)
    joined = df_points.crossJoin(df_locais_unicos.hint("broadcast")) \
        .withColumn("dist", haversine_udf(F.col(lat_col), F.col(lng_col), F.col("latitude"), F.col("longitude")))
    
    # Seleciona o ponto com a menor distância para cada local original
    from pyspark.sql.window import Window
    window_spec = Window.partitionBy(lat_col, lng_col).orderBy("dist")
    
    return joined.withColumn("row", F.row_number().over(window_spec)) \
        .filter(F.col("row") == 1) \
        .select(
            F.col(lat_col),
            F.col(lng_col),
            F.col("latitude").alias(f"{prefix}_lat_join"),
            F.col("longitude").alias(f"{prefix}_lng_join")
        )

# 3. Mapear as coordenadas meteorológicas mais próximas
customer_map = find_nearest_weather(customer_locs, "customer_geolocation_lat", "customer_geolocation_lng", "customer")
seller_map = find_nearest_weather(seller_locs, "seller_geolocation_lat", "seller_geolocation_lng", "seller")

# 4. Trazer as colunas de volta para o DataFrame de Orders original
orders_final = orders \
    .join(customer_map, ["customer_geolocation_lat", "customer_geolocation_lng"], "left") \
    .join(seller_map, ["seller_geolocation_lat", "seller_geolocation_lng"], "left") \
    .drop(
        "customer_geolocation_lat", 
        "customer_geolocation_lng", 
        "seller_geolocation_lat", 
        "seller_geolocation_lng"
    )

# Visualizar o resultado
orders_final.show()

+----------------------------+-------------------------------+-----------------------------+--------------------------------+-------------------+-------------------+--------------------+------------------+-------------+--------------------+----------+------+-------------+----------------+----------+-----------+---------+------------+-----------------+-----------------+-----------------+---------------+---------------+
|order_delivered_carrier_date|interval_code_delivered_carrier|order_delivered_customer_date|interval_code_delivered_customer|approval_time_hours|handling_time_hours|shipping_delay_hours|delivery_time_days|purchase_hour|purchase_day_of_week|is_weekend| price|freight_value|product_weight_g|volume_cm3|distance_km|same_city|review_score|delivered_on_time|customer_lat_join|customer_lng_join|seller_lat_join|seller_lng_join|
+----------------------------+-------------------------------+-----------------------------+--------------------------------+-------------------+-----------

In [8]:
from pyspark.sql import functions as F

# 1. Preparação das datas na tabela orders (garantir que são apenas datas sem hora para o join)
orders_prep = orders_final \
    .withColumn("date_carrier", F.to_date("order_delivered_carrier_date")) \
    .withColumn("date_customer", F.to_date("order_delivered_customer_date"))

# 2. Join para os dados do SELLER
# Usamos a data da transportadora e o local do vendedor
weather_seller = weather.select(
    F.col("date").alias("date_carrier"),
    F.col("interval_code").alias("interval_code_delivered_carrier"),
    F.col("latitude").alias("seller_lat_join"),
    F.col("longitude").alias("seller_lng_join"),
    F.col("total_rainfall_period_mm").alias("seller_total_rainfall_period_mm"),
    F.col("max_rain_intensity_mm_h").alias("seller_max_rain_intensity_mm_h"),
    F.col("max_wind_gust_period_ms").alias("seller_max_wind_gust_period_ms"),
    F.col("interval_code").alias("seller_interval_code"),
    F.col("rain_class_code").alias("seller_rain_class_code"),
    F.col("wind_class_code").alias("seller_wind_class_code"),
    F.col("month").alias("month"),
    F.col("region").alias("seller_region"),
    F.col("accumulated_rainfall_3_days_mm").alias("seller_accumulated_rainfall_3_days_mm"),
    F.col("is_heavy_rain").alias("seller_is_heavy_rain"),
    F.col("is_strong_wind").alias("seller_is_strong_wind")
)

orders_with_seller_weather = orders_prep.join(
    weather_seller, 
    on=["date_carrier", "interval_code_delivered_carrier", "seller_lat_join", "seller_lng_join"], 
    how="left"
)

# 3. Join para os dados do CUSTOMER
# Usamos a data de entrega e o local do cliente
weather_customer = weather.select(
    F.col("date").alias("date_customer"),
    F.col("interval_code").alias("interval_code_delivered_customer"),
    F.col("latitude").alias("customer_lat_join"),
    F.col("longitude").alias("customer_lng_join"),
    F.col("total_rainfall_period_mm").alias("customer_total_rainfall_period_mm"),
    F.col("max_rain_intensity_mm_h").alias("customer_max_rain_intensity_mm_h"),
    F.col("max_wind_gust_period_ms").alias("customer_max_wind_gust_period_ms"),
    F.col("interval_code").alias("customer_interval_code"),
    F.col("rain_class_code").alias("customer_rain_class_code"),
    F.col("wind_class_code").alias("customer_wind_class_code"),
    F.col("region").alias("customer_region"),
    F.col("accumulated_rainfall_3_days_mm").alias("customer_accumulated_rainfall_3_days_mm"),
    F.col("is_heavy_rain").alias("customer_is_heavy_rain"),
    F.col("is_strong_wind").alias("customer_is_strong_wind")

)

df_final = orders_with_seller_weather.join(
    weather_customer, 
    on=["date_customer", "interval_code_delivered_customer", "customer_lat_join", "customer_lng_join"], 
    how="left"
)

# 4. Seleção final das colunas e renomeação conforme sua lista
# Note: Renomeei as colunas de join de volta para os nomes originais como você pediu no esquema final
df_final_selected = df_final.select(
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "approval_time_hours",
    "handling_time_hours",
    "shipping_delay_hours",
    "delivery_time_days",
    "purchase_hour",
    "purchase_day_of_week",
    "is_weekend",
    "price",
    "freight_value",
    "product_weight_g",
    "volume_cm3",
    "distance_km",
    "same_city",
    "review_score",
    "delivered_on_time",
    F.col("seller_lat_join").alias("seller_geolocation_lat"),
    F.col("seller_lng_join").alias("seller_geolocation_lng"),
    "seller_total_rainfall_period_mm",
    "seller_max_rain_intensity_mm_h",
    "seller_max_wind_gust_period_ms",
    "seller_interval_code",
    "seller_rain_class_code",
    "seller_wind_class_code",
    F.col("customer_lat_join").alias("customer_geolocation_lat"),
    F.col("customer_lng_join").alias("customer_geolocation_lng"),
    "customer_total_rainfall_period_mm",
    "customer_max_rain_intensity_mm_h",
    "customer_max_wind_gust_period_ms",
    "customer_interval_code",
    "customer_rain_class_code",
    "customer_wind_class_code",
    "month",
    "seller_region",
    "customer_region",
    "seller_accumulated_rainfall_3_days_mm",
    "seller_is_heavy_rain",
    "seller_is_strong_wind",
    "customer_accumulated_rainfall_3_days_mm",
    "customer_is_heavy_rain",
    "customer_is_strong_wind"
)

df_final_selected.show()

26/04/19 03:18:09 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------------------+-----------------------------+-------------------+-------------------+--------------------+------------------+-------------+--------------------+----------+-----+-------------+----------------+----------+-----------+---------+------------+-----------------+----------------------+----------------------+-------------------------------+------------------------------+------------------------------+--------------------+----------------------+----------------------+------------------------+------------------------+---------------------------------+--------------------------------+--------------------------------+----------------------+------------------------+------------------------+-----+--------------------+--------------------+-------------------------------------+--------------------+---------------------+---------------------------------------+----------------------+-----------------------+
|order_delivered_carrier_date|order_delivered_customer_date|approv

In [9]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

# # --- 1. CARREGAMENTO DOS DADOS ---
# orders = spark.read.csv("s3a://last-mile-optimization-trusted/dataset-orders/final_table_orders_numeric.csv/", header=True, inferSchema=True)
# weather = spark.read.csv("s3a://last-mile-optimization-trusted/dataset-weather/", header=True, inferSchema=True)

# Coordenadas únicas da tabela de clima (Estações)
df_locais_unicos = weather.select("latitude", "longitude").distinct().cache()

# --- 2. PRÉ-CÁLCULO DOS 5 VIZINHOS MAIS PRÓXIMOS ---
# Usamos Haversine para mapear quais estações podem substituir as que estão sem dados
df_vizinhos = (
    df_locais_unicos.alias("a").crossJoin(df_locais_unicos.alias("b"))
    .withColumn("dlat", F.radians(F.col("b.latitude") - F.col("a.latitude")))
    .withColumn("dlon", F.radians(F.col("b.longitude") - F.col("a.longitude")))
    .withColumn("dist_calc", 
        F.sin(F.col("dlat")/2)**2 + 
        F.cos(F.radians(F.col("a.latitude"))) * F.cos(F.radians(F.col("b.latitude"))) * F.sin(F.col("dlon")/2)**2
    )
    .withColumn("distancia_km", F.lit(2) * 6371 * F.asin(F.sqrt(F.col("dist_calc"))))
)

window_v = Window.partitionBy("a.latitude", "a.longitude").orderBy("distancia_km")

# Top 5 vizinhos (excluindo a si mesma, rank > 1)
df_top_vizinhos = df_vizinhos.withColumn("rank", F.row_number().over(window_v)) \
    .filter((F.col("rank") > 1) & (F.col("rank") <= 8)) \
    .select(
        F.col("a.latitude").alias("lat_orig"), 
        F.col("a.longitude").alias("lon_orig"), 
        F.col("b.latitude").alias("lat_vizinho"), 
        F.col("b.longitude").alias("lon_vizinho"), 
        F.col("rank")
    ).cache()

# --- 3. PREPARAÇÃO DA TABELA ORDERS ---
# Adicionando datas e nomes padronizados para o Join
orders_prep = orders_final \
    .withColumn("date_seller", F.to_date("order_delivered_carrier_date")) \
    .withColumn("date_customer", F.to_date("order_delivered_customer_date")) \
    .withColumnRenamed("customer_geolocation_lat_join", "customer_lat_join") \
    .withColumnRenamed("customer_geolocation_lng_join", "customer_lng_join") \
    .withColumnRenamed("seller_geolocation_lat_join", "seller_lat_join") \
    .withColumnRenamed("seller_geolocation_lng_join", "seller_lng_join")

# --- 4. FUNÇÃO DE JOIN COM TRATAMENTO DE NULOS (BACKFILL) ---
def join_weather_with_fallback(base_df, weather_df, prefix):
    cols_weather = [
        "total_rainfall_period_mm",
        "max_rain_intensity_mm_h",
        "max_wind_gust_period_ms",
        "interval_code",
        "rain_class_code",
        "wind_class_code",
        "accumulated_rainfall_3_days_mm",
        "is_heavy_rain",
        "is_strong_wind",
        "region",
        "month"
    ]
    
    # 4.1. Join com a Estação Original
    w_orig = weather_df.select([F.col(c).alias(f"orig_{c}") for c in cols_weather] + ["date", "latitude", "longitude", "interval_code"])
    
    res = base_df.join(
        w_orig,
        (base_df[f"{prefix}_lat_join"] == w_orig["latitude"]) & 
        (base_df[f"{prefix}_lng_join"] == w_orig["longitude"]) &
        (base_df[f"date_{prefix}"] == w_orig["date"]) &
        (base_df[f"interval_code_delivered_{'carrier' if prefix=='seller' else 'customer'}"] == w_orig["interval_code"]),
        "left"
    ).drop("latitude", "longitude", "date", "interval_code")

    # 4.2. Loop para tentar preencher com os 5 vizinhos se o original for NULL
    for r in range(2, 7): # Rank 2 a 6 (os 5 vizinhos)
        v_map = df_top_vizinhos.filter(F.col("rank") == r).alias(f"vmap_{r}")
        
        # Pega a coordenada do vizinho
        res = res.join(v_map, 
            (res[f"{prefix}_lat_join"] == v_map["lat_orig"]) & 
            (res[f"{prefix}_lng_join"] == v_map["lon_orig"]), "left")
        
        # Pega o clima do vizinho
        w_viz = weather_df.select([F.col(c).alias(f"v{r}_{c}") for c in cols_weather] + ["date", "latitude", "longitude", "interval_code"])
        
        res = res.join(w_viz,
            (res["lat_vizinho"] == w_viz["latitude"]) & 
            (res["lon_vizinho"] == w_viz["longitude"]) &
            (res[f"date_{prefix}"] == w_viz["date"]) &
            (res[f"interval_code_delivered_{'carrier' if prefix=='seller' else 'customer'}"] == w_viz["interval_code"]),
            "left"
        ).drop("latitude", "longitude", "date", "interval_code", "lat_orig", "lon_orig", "lat_vizinho", "lon_vizinho", "rank")

        # COALESCE: Mantém o valor se já existir, se for null, pega o do vizinho
        for c in cols_weather:
            res = res.withColumn(f"orig_{c}", F.coalesce(F.col(f"orig_{c}"), F.col(f"v{r}_{c}")))
            res = res.drop(f"v{r}_{c}")

    # Renomeação final
    final_cols = []
    for c in cols_weather:
        if c == "month":
            if "month" in res.columns:
                res = res.withColumn("month", F.coalesce(F.col("month"), F.col("orig_month")))
            else:
                res = res.withColumn("month", F.col("orig_month"))
            final_cols.append("orig_month")
        elif c == "region":
            res = res.withColumn(f"{prefix}_region", F.col("orig_region"))
            final_cols.append("orig_region")
        else:
            res = res.withColumn(f"{prefix}_{c}", F.col(f"orig_{c}"))
            final_cols.append(f"orig_{c}")
    
    return res.drop(*final_cols)

# --- 5. EXECUÇÃO E SELEÇÃO FINAL ---
df_with_seller = join_weather_with_fallback(orders_prep, weather, "seller")
df_final = join_weather_with_fallback(df_with_seller, weather, "customer")

# Colunas finais conforme solicitado
df_resultado = df_final.select(
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "approval_time_hours", "handling_time_hours", "shipping_delay_hours",
    "delivery_time_days", "purchase_hour", "purchase_day_of_week", "is_weekend",
    "price", "freight_value", "product_weight_g", "volume_cm3", "distance_km",
    "same_city", "review_score", "delivered_on_time",
    F.col("seller_lat_join").alias("seller_geolocation_lat"),
    F.col("seller_lng_join").alias("seller_geolocation_lng"),
    "seller_total_rainfall_period_mm", "seller_max_rain_intensity_mm_h",
    "seller_max_wind_gust_period_ms", "seller_interval_code",
    "seller_rain_class_code", "seller_wind_class_code",
    F.col("customer_lat_join").alias("customer_geolocation_lat"),
    F.col("customer_lng_join").alias("customer_geolocation_lng"),
    "customer_total_rainfall_period_mm", "customer_max_rain_intensity_mm_h",
    "customer_max_wind_gust_period_ms", "customer_interval_code",
    "customer_rain_class_code", "customer_wind_class_code",
    "month",
    "seller_region",
    "customer_region",
    "seller_accumulated_rainfall_3_days_mm",
    "seller_is_heavy_rain",
    "seller_is_strong_wind",
    "customer_accumulated_rainfall_3_days_mm",
    "customer_is_heavy_rain",
    "customer_is_strong_wind"
    
)

df_resultado.show()

+----------------------------+-----------------------------+-------------------+-------------------+--------------------+------------------+-------------+--------------------+----------+------+-------------+----------------+----------+-----------+---------+------------+-----------------+----------------------+----------------------+-------------------------------+------------------------------+------------------------------+--------------------+----------------------+----------------------+------------------------+------------------------+---------------------------------+--------------------------------+--------------------------------+----------------------+------------------------+------------------------+-----+--------------------+--------------------+-------------------------------------+--------------------+---------------------+---------------------------------------+----------------------+-----------------------+
|order_delivered_carrier_date|order_delivered_customer_date|appro

In [10]:
from pyspark.sql import functions as F

# Conta os nulos para cada coluna do DataFrame
null_counts = df_resultado.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_resultado.columns
])

# Exibe o resultado
null_counts.show(vertical=True)

-RECORD 0--------------------------------------
 order_delivered_carrier_date            | 0   
 order_delivered_customer_date           | 0   
 approval_time_hours                     | 0   
 handling_time_hours                     | 0   
 shipping_delay_hours                    | 0   
 delivery_time_days                      | 0   
 purchase_hour                           | 0   
 purchase_day_of_week                    | 0   
 is_weekend                              | 0   
 price                                   | 0   
 freight_value                           | 0   
 product_weight_g                        | 4   
 volume_cm3                              | 4   
 distance_km                             | 0   
 same_city                               | 0   
 review_score                            | 218 
 delivered_on_time                       | 0   
 seller_geolocation_lat                  | 0   
 seller_geolocation_lng                  | 0   
 seller_total_rainfall_period_mm        

In [11]:
# 1. Pegamos a lista de todas as colunas, exceto 'review_score'
colunas_para_verificar = [c for c in df_resultado.columns if c != 'review_score']

# 2. Removemos as linhas que possuem nulos nessas colunas específicas
df_limpo = df_resultado.dropna(subset=colunas_para_verificar)

# 3. Verificamos quantas linhas restaram
print(f"Total de linhas antes: {df_resultado.count()}")
print(f"Total de linhas após a limpeza: {df_limpo.count()}")

# Exibe o resultado final
df_limpo.show()

Total de linhas antes: 33824


Total de linhas após a limpeza: 33820


+----------------------------+-----------------------------+-------------------+-------------------+--------------------+------------------+-------------+--------------------+----------+------+-------------+----------------+----------+-----------+---------+------------+-----------------+----------------------+----------------------+-------------------------------+------------------------------+------------------------------+--------------------+----------------------+----------------------+------------------------+------------------------+---------------------------------+--------------------------------+--------------------------------+----------------------+------------------------+------------------------+-----+--------------------+--------------------+-------------------------------------+--------------------+---------------------+---------------------------------------+----------------------+-----------------------+
|order_delivered_carrier_date|order_delivered_customer_date|appro

In [12]:
# from pyspark.sql import functions as F

# # Conta os nulos para cada coluna do DataFrame
# null_counts = df_limpo.select([
#     F.count(F.when(F.col(c).isNull(), c)).alias(c) 
#     for c in df_limpo.columns
# ])

# # Exibe o resultado
# null_counts.show(vertical=True)

-RECORD 0--------------------------------------
 order_delivered_carrier_date            | 0   
 order_delivered_customer_date           | 0   
 approval_time_hours                     | 0   
 handling_time_hours                     | 0   
 shipping_delay_hours                    | 0   
 delivery_time_days                      | 0   
 purchase_hour                           | 0   
 purchase_day_of_week                    | 0   
 is_weekend                              | 0   
 price                                   | 0   
 freight_value                           | 0   
 product_weight_g                        | 0   
 volume_cm3                              | 0   
 distance_km                             | 0   
 same_city                               | 0   
 review_score                            | 218 
 delivered_on_time                       | 0   
 seller_geolocation_lat                  | 0   
 seller_geolocation_lng                  | 0   
 seller_total_rainfall_period_mm        

In [13]:
%pip install scikit-learn

from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder

Note: you may need to restart the kernel to use updated packages.


In [14]:
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array # Importação crucial para Spark 3.x

# 1. Definir o dicionário de mesorregiões
meso_dict = {
    'LINS': 'BAURU', 'CACHOEIRA PAULISTA': 'VALE DO PARAIBA PAULISTA',
    'BRAGANCA PAULISTA': 'MACRO METROPOLITANA PAULISTA', 'SAO LUIZ DO PARAITINGA': 'VALE DO PARAIBA PAULISTA',
    'PRESIDENTE PRUDENTE': 'PRESIDENTE PRUDENTE', 'SAO PAULO - MIRANTE': 'METROPOLITANA DE SAO PAULO',
    'SAO CARLOS': 'ARARAQUARA', 'BERTIOGA': 'METROPOLITANA DE SAO PAULO',
    'REGISTRO': 'LITORAL SUL PAULISTA', 'ITAPEVA': 'ITAPETININGA',
    'BARRA BONITA': 'BAURU', 'JOSE BONIFACIO': 'SAO JOSE DO RIO PRETO',
    'CASA BRANCA': 'CAMPINAS', 'BARUERI': 'METROPOLITANA DE SAO PAULO',
    'IBITINGA': 'ARARAQUARA', 'BARRETOS': 'RIBEIRAO PRETO',
    'ARIRANHA': 'SAO JOSE DO RIO PRETO', 'ITUVERAVA': 'RIBEIRAO PRETO',
    'SAO PAULO - INTERLAGOS': 'METROPOLITANA DE SAO PAULO', 'CAMPOS DO JORDAO': 'VALE DO PARAIBA PAULISTA',
    'FRANCA': 'RIBEIRAO PRETO', 'SAO SIMAO': 'RIBEIRAO PRETO',
    'OURINHOS': 'ASSIS', 'TAUBATE': 'VALE DO PARAIBA PAULISTA',
    'BARRA DO TURVO': 'LITORAL SUL PAULISTA', 'IGUAPE': 'LITORAL SUL PAULISTA',
    'IPERO': 'MACRO METROPOLITANA PAULISTA', 'DRACENA': 'PRESIDENTE PRUDENTE',
    'PRADOPOLIS': 'RIBEIRAO PRETO', 'BAURU': 'BAURU',
    'AVARE': 'BAURU', 'RANCHARIA': 'PRESIDENTE PRUDENTE',
    'VALPARAISO': 'ARACATUBA', 'SAO MIGUEL ARCANJO': 'MACRO METROPOLITANA PAULISTA',
    'MARILIA': 'MARILIA', 'BEBEDOURO': 'RIBEIRAO PRETO',
    'PIRACICABA': 'PIRACICABA', 'ITAPIRA': 'CAMPINAS',
    'JALES': 'SAO JOSE DO RIO PRETO', 'TUPA': 'MARILIA'
}

# 2. Criar as colunas de mesorregião e aplicar o replace
df_com_meso = df_limpo.withColumn("seller_meso", col("seller_region")) \
                      .withColumn("customer_meso", col("customer_region"))

df_com_meso = df_com_meso.replace(to_replace=meso_dict, subset=['seller_meso', 'customer_meso'])

# 3. Pipeline de Encoding
categorical_cols = ['seller_meso', 'customer_meso']
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") for c in categorical_cols]
encoder = OneHotEncoder(inputCols=[f"{c}_index" for c in categorical_cols], 
                        outputCols=[f"{c}_vec" for c in categorical_cols],
                        dropLast=False)

pipeline = Pipeline(stages=indexers + [encoder])
model = pipeline.fit(df_com_meso)
df_vectorized = model.transform(df_com_meso)

# 4. Função otimizada para "explodir" o vetor em colunas binárias
def expand_ohe_columns(df, input_col, labels):
    # Converte o vetor em um Array para que possamos acessar os índices
    df = df.withColumn("temp_array", vector_to_array(col(f"{input_col}_vec")))
    
    # Cria uma lista de novas colunas (0/1) baseada nos labels
    new_columns = []
    for i, label in enumerate(labels):
        clean_label = label.replace(" ", "_").replace("-", "_").lower()
        col_name = f"{input_col}_{clean_label}"
        new_columns.append(col("temp_array").getItem(i).cast("int").alias(col_name))
    
    # Adiciona todas as colunas de uma vez e remove a temporária
    return df.select("*", *new_columns).drop("temp_array")

# Pegar os labels para nomear as colunas
labels_seller = model.stages[0].labels
labels_customer = model.stages[1].labels

# Aplicar a expansão
df_final = expand_ohe_columns(df_vectorized, "seller_meso", labels_seller)
df_final = expand_ohe_columns(df_final, "customer_meso", labels_customer)

# 5. Limpeza: Remover colunas de índices e vetores intermediários
cols_to_drop = ['seller_meso_index', 'customer_meso_index', 'seller_meso_vec', 'customer_meso_vec']
df_final = df_final.drop(*cols_to_drop)

# Mostrar o resultado final com as novas colunas
df_final.select("seller_region", "seller_meso", *[c for c in df_final.columns if "seller_meso_" in c]).show(5)

+--------------------+--------------------+--------------------------------------+----------------------+----------------------+--------------------------+---------------------------------+--------------------+------------------------------------+----------------------------------------+-----------------+-------------------------------+-------------------+-----------------+---------------------+--------------------------------+------------------------+
|       seller_region|         seller_meso|seller_meso_metropolitana_de_sao_paulo|seller_meso_araraquara|seller_meso_piracicaba|seller_meso_ribeirao_preto|seller_meso_sao_jose_do_rio_preto|seller_meso_campinas|seller_meso_vale_do_paraiba_paulista|seller_meso_macro_metropolitana_paulista|seller_meso_bauru|seller_meso_presidente_prudente|seller_meso_marilia|seller_meso_assis|seller_meso_aracatuba|seller_meso_litoral_sul_paulista|seller_meso_itapetininga|
+--------------------+--------------------+--------------------------------------+----

In [15]:
# df_final.show()

+----------------------------+-----------------------------+-------------------+-------------------+--------------------+------------------+-------------+--------------------+----------+------+-------------+----------------+----------+-----------+---------+------------+-----------------+----------------------+----------------------+-------------------------------+------------------------------+------------------------------+--------------------+----------------------+----------------------+------------------------+------------------------+---------------------------------+--------------------------------+--------------------------------+----------------------+------------------------+------------------------+-----+--------------------+--------------------+-------------------------------------+--------------------+---------------------+---------------------------------------+----------------------+-----------------------+--------------------+--------------------+------------------------

In [16]:
df_final = df_final.drop('seller_meso', 'customer_meso', 'seller_region', 'customer_region')

In [17]:
df_final.show()

+----------------------------+-----------------------------+-------------------+-------------------+--------------------+------------------+-------------+--------------------+----------+------+-------------+----------------+----------+-----------+---------+------------+-----------------+----------------------+----------------------+-------------------------------+------------------------------+------------------------------+--------------------+----------------------+----------------------+------------------------+------------------------+---------------------------------+--------------------------------+--------------------------------+----------------------+------------------------+------------------------+-----+-------------------------------------+--------------------+---------------------+---------------------------------------+----------------------+-----------------------+--------------------------------------+----------------------+----------------------+-----------------------

In [18]:
df_final.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-join/join_orders_and_weather.csv')

spark.stop()

26/04/19 03:26:31 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/19 03:26:32 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
